<div style="text-align:center;">
  <span style="color:#ff00d4; font-weight:bold; font-size:40px;">
    250528 Collision Test
  </span>
</div>

<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Import Library
</span>

In [ ]:
import rby1_sdk as rby
from dynamixel_sdk import PortHandler, PacketHandler
import time
import numpy as np
import matplotlib as plt
import logging
import sys
from pathlib import Path

# 0: folder_a/folder_b-file.py -> sfolder_a / 1: folder_a/folder_b/folder_c-file.py -> folder_a
sys.path.append(str(Path.cwd().parents[0]))                         # .ipynb
# sys.path.append(str(Path(__file__).resolve().parents[0]))           # .py
import functions as fn

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

SIM_OR_REAL = 0         # 0: Docker Simulation / 1: Real Robot(Wifi Remote Access mode) / 2: Real Robot (UPC Direct mode)

<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Power ON
</span>

In [4]:
if (SIM_OR_REAL == 0):
    ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
elif (SIM_OR_REAL == 1 or SIM_OR_REAL == 2):
    ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot
else:
    ROBOT_ADDRESS = "localhost:50051"           # For Safety, Default=Docker Simulation

robot = rby.create_robot_a(ROBOT_ADDRESS)       # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("Could not connect to robot")
    exit(1)
else:
    print("SUCCESS: Connect to the robot.")
    if (robot.power_on(".*") == True):             # Turn on all the power (5v, 12v, 24v, 48v)
        time.sleep(1)
        if (robot.servo_on(".*") == True):         # Enable servo motors on all actuators
            time.sleep(1)
            robot.reset_fault_control_manager()
            robot.enable_control_manager(unlimited_mode_enabled=True)
            print("SUCCESS: Power ON the Robot.")
            
            # Grippers 12V Power ON 
            time.sleep(1)
            robot.set_tool_flange_output_voltage("right", 12)
            robot.set_tool_flange_output_voltage("left", 12)
            print("SUCCESS: Power ON the Gripper.")
        else:
            print("Failed to power ON the Servo")

SUCCESS: Connect to the robot.
SUCCESS: Power ON the Robot.
SUCCESS: Power ON the Gripper.


<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Power OFF
</span>

In [3]:
robot.set_tool_flange_output_voltage("right", 0)
robot.set_tool_flange_output_voltage("left", 0)
time.sleep(1)
robot.disable_control_manager()
robot.power_off(".*")

True

<span style="color:#5ea6ff; font-weight:bold; font-size:30px;">
Test for Robot State
</span>

In [5]:
import inspect

robot_state = robot.get_state()
robot_state_properties = [name for name, obj in
                          inspect.getmembers(type(robot_state), lambda x: isinstance(x, property))]
print(f"Available Robot States:\n {robot_state_properties}")

Available Robot States:
 ['battery_state', 'center_of_mass', 'collisions', 'current', 'emo_states', 'ft_sensor_left', 'ft_sensor_right', 'gravity', 'is_ready', 'joint_states', 'odometry', 'position', 'power_states', 'system_stat', 'target_feedback_gain', 'target_feedforward_torque', 'target_position', 'target_velocity', 'temperature', 'timestamp', 'tool_flange_left', 'tool_flange_right', 'torque', 'velocity']


<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Posture Change
</span>

In [5]:
Desired_Pos = fn.ZERO_POS                             # ZERO_POS, READY_POS, PACK_POS, READY_DEMO_POS
fn.move_position_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)
time.sleep(1)

<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Functions
</span>

In [ ]:
COLLISION_THRESHOLD = 0.05      # [m]
RIGHT_ARM_TARGET = np.array([0.8, -0.8, 0.7, -2.0, 0, 0.0, 0])  # [rad]
canceling_time_param = "joint_position_command.canceling_time"
canceling_duration_list = [0.1, 0.5]        # [sec] Time duration from start canceling to real stop
MINIMUM_TIME = 5.0


def move_pre_control_pose(robot):
    """Move to the pre-control pose before starting the motion."""
    init_pose = {"torso":     np.array([0.0, 0.1, -0.2, 0.1, 0.0, 0.0]),
            "right_arm": np.array([0.2, -0.2, 0.0, -1.0, 0, 0.7, 0.0]),
            "left_arm":  np.array([0.2, 0.2, 0.0, -1.0, 0, 0.7, 0.0])}
    if not fn.move_position_20dof(robot, posture=init_pose, minimum_time=5.0, control_hold_time=0.5, get_return=True):
        exit(1)
        
        
# "joint_position_command.canceling_time"이 사용 가능한지 확인하는 코드
def validate_parameter(robot, parameter_name):
    available = {name for name, _ in robot.get_parameter_list()}
    if parameter_name not in available:
        raise RuntimeError(
            f"Parameter '{parameter_name}' is not available on this robot. "
            f"Available canceling_time parameters: "
            f"{sorted(name for name in available if name.endswith('.canceling_time'))}"
        )
        

class CollisionTestRecorder:
    def __init__(self, robot, record_freq=50):
        self.robot = robot
        self.record_freq = record_freq
        self.time_sec = []
        self.nearest_links = []
        self.nearest_distance = []
        self.start_time = None
        self.stop_requested = False
        
    def __enter__(self):
        self.time_sec = []
        self.nearest_links = []
        self.nearest_distance = []
        self.start_time = time.monotonic()
        self.robot.stop_state_update()
        self.robot.start_state_update(self.cb, rate=self.record_freq)
        return self
    
    def __exit__(self):
        self.robot.stop_state_update()
            
    def cb(self, robot_state, control_manager_state):
        del control_manager_state
        elapsed_time = time.monotonic() - self.start_time
        self.time_sec.append(elapsed_time)
        if not robot_state.collisions:
            self.nearest_links.append([None, None])
            self.nearest_distance.append(None)
        else:
            nearest_obj = min(robot_state.collisions, key=lambda collision: collision.distance)
            self.nearest_links.append((nearest_obj.link1, nearest_obj.link2))
            self.nearest_distance.append(nearest_obj.distance)
        
        print(
            f"nearest collision distance: {nearest_obj.distance:.4f}[m] | "
            f"{nearest_obj.link1} <-> {nearest_obj.link2}"
        )
        if nearest_obj.distance < COLLISION_THRESHOLD:
            print(f"Collision Threshold Detected")
            self.stop_requested = True
            
    def return_results(self):
        return {
            "time_sec": np.asarray(self.time_sec, dtype=float),                     # (N,)
            "nearest_links": np.asarray(self.nearest_links, dtype=str),             # (N,2)
            "nearest_distance": np.asarray(self.nearest_distance, dtype=float)      # (N,)
        }
        

def run_collisionTest(robot, canceling_time_param, requested_canceling_duration, minimum_time):
    if not robot.set_parameter(canceling_time_param, f"{requested_canceling_duration}"):
        raise RuntimeError(f"Failed to set {canceling_time_param}")
    
    t0 = time.monotonic()
    control_cancel_time = None
    control_stop_time   = None
        
    collision_pose = {"torso":     None,
                      "right_arm": RIGHT_ARM_TARGET,
                      "left_arm":  None}
    
    with CollisionTestRecorder(robot) as recorder:
        rc = fn.move_position_20dof(robot, posture=collision_pose, minimum_time=minimum_time, control_hold_time=0, get_return=False)  # get_return=False
        while True:
            elapsed_time = time.monotonic() - t0
            if (recorder.stop_requested == True):
                robot.cancel_control()
                control_cancel_time = elapsed_time
            if rc.wait_for(10):
                control_stop_time = time.monotonic() - t0
                break;
            
    if control_stop_time == None:
        raise RuntimeError("Failed to observe command completion")
    
    samples = recorder.return_results()
    if samples["time_sec"].shape[0] == 0:
        raise RuntimeError("Failed to record joint trajectory")
        
    return {
        "canceling_time_param": canceling_time_param,
        "requested_cancel_duration": requested_canceling_duration,
        "control_cancel_time": control_cancel_time,
        "control_stop_time": control_stop_time,
        "cancel_to_stop_duration": control_stop_time - control_cancel_time,
        "samples": samples
    }


def plot_results(results):
    plt.rcParams.update({"font.size": 12})

    fig, axes = plt.subplots(len(results), 1, sharex=True, sharey=True, figsize=(12, 4 * len(results)))
    axes = np.atleast_1d(axes)

    all_trials = []
    for result in results:
        samples = result["samples"]
        all_trials.append([samples["time_sec"], samples["nearest_distance"]])

    if all_trials:
        x_max = max(np.max(trial[0]) for trial in all_trials)
        
        y_min = min(np.min(trial[1]) for trial in all_trials)
        y_max = max(np.max(trial[1]) for trial in all_trials)
        y_pad = max(1.0, 0.08 * (y_max - y_min))
        y_min -= y_pad
        y_max += y_pad
    else:
        x_max = MINIMUM_TIME
        y_min, y_max = -1.0, 1.0

    for i, result in enumerate(results):
        samples = result["samples"]
        
        time_sec = samples["time_sec"]
        nearest_links = samples["nearest_links"]
        nearest_distance = samples["nearest_distance"]
        
        requested_cancel_duration = result["requested_cancel_duration"]
        control_cancel_time = result["control_cancel_time"]
        control_stop_time = result["control_stop_time"]
        cancel_to_stop_duration = result["cancel_to_stop_duration"]

        axes[i].set_title(
            f"Requested Canceling Duration: {requested_cancel_duration:.2f}s\n"
            f"Minimum Time: {MINIMUM_TIME:.1f}s")
        axes[i].plot(time_sec, nearest_distance, color='r', linewidth=2.0, label='Nearest Distance[m]')
        axes[i].axvline(x=control_cancel_time, color="k", linewidth=2.0, linestyle="--", label='Control Cancel Time')
        axes[i].axvline(x=control_stop_time, color="g", linewidth=2.0, linestyle="--", label='Control Stop Time')
        axes[i].text(0.02, 0.95, f"Cancel Duration: {cancel_to_stop_duration:.2f}s", transform=axes[i].transAxes, va="top", ha="left")
        # link pair가 바뀌는 지점만 표시
        prev_pair = None
        for t, d, pair in zip(time_sec, nearest_distance, nearest_links):
            link1, link2 = pair

            if link1 in ["None", "", None] or link2 in ["None", "", None]:
                continue

            if np.isnan(d):
                continue

            current_pair = (link1, link2)

            if current_pair == prev_pair:
                continue

            label = f"{link1}\n↔\n{link2}"

            axes[i].annotate(
                label,
                xy=(t, d),
                xytext=(8, 12),
                textcoords="offset points",
                fontsize=8,
                arrowprops=dict(arrowstyle="->", linewidth=0.8),
            )

            prev_pair = current_pair
            
        axes[i].legend(loc="lower right")
        axes[i].set_xlim([0, x_max])
        axes[i].set_ylim([y_min, y_max])

        if i != len(results) - 1:
            axes[i].get_xaxis().set_visible(False)

    plt.subplots_adjust(hspace=0.4)
    fig.supxlabel("Time [sec]")
    fig.supylabel("Nearest Distance [m]")

    plt.show()
    

<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Run Collision Test
</span>

In [ ]:
print("===== Collision Test =====")
validate_parameter(robot, canceling_time_param)                 # "joint_position_command.canceling_time"
original_value = robot.get_parameter(canceling_time_param)      # Default original canceling duration value

print(f"Using canceling_time parameter: {canceling_time_param}")
print(f"Original value: {original_value}")


# Set Initial Posture
move_pre_control_pose(robot)
time.sleep(1)

results = []

try:
    for idx, canceling_duration in enumerate(canceling_duration_list):
        print("Running trial with canceling_duration=%.2f s", canceling_duration)
        result = run_collisionTest(
            robot=robot,
            canceling_time_param=canceling_time_param,
            requested_canceling_duration=canceling_duration,
            minimum_time=MINIMUM_TIME,
        )
        results.append(result)

        if not robot.set_parameter(canceling_time_param, original_value):
            raise RuntimeError(f"Failed to restore {canceling_time_param}")

        if idx < len(canceling_duration_list) - 1:
            move_pre_control_pose(robot)    # Set Initial Posture
            time.sleep(1)
            
    plot_results(results)
finally:
    try:
        robot.set_parameter(canceling_time_param, original_value)
    except Exception:
        pass

    try:
        robot.cancel_control()
    except Exception:
        pass

    try:
        move_pre_control_pose(robot)    # Set Initial Posture
        time.sleep(1)
    except Exception:
        pass